<a href="https://colab.research.google.com/github/raz0208/Natural-Language-Processing-Practices/blob/main/TopicModelling/EmbeddingsAnalysis_TopicClustering_ModernBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topic Clustring

### Clustering Topic Models from TurfTopic

1. TurfTopic Default model and configuration
2. Use ModernBert for embeddings

In [ ]:
# Install libraries and packages
!pip install 'turftopic[umap-learn, datamapplot]'

In [22]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch

In [23]:
# Read and Load dataset
dataset = pd.read_csv('sample_PubMedDataAbstracts.csv')

# Show the datasets
### Abstract Embeddings Sample Dataset
print('Node Content:', dataset.shape)
print(dataset)

Node Content: (10000, 4)
      Unnamed: 0                                              title  \
0              0  Phenotypic variability of Niemann-Pick disease...   
1              1  Recurrent hypoglycemia secondary to metformin ...   
2              2  Adaptation of the Ambulatory and Home Care Rec...   
3              3  Multidimensional family therapy in adolescents...   
4              4  Balanced crystalloids versus isotonic saline i...   
...          ...                                                ...   
9995        9995  Methylmercury in Industrial Harbor Sediments i...   
9996        9996  Factors Affecting Secondhand Smoke Avoidance B...   
9997        9997  Predicting Infectious Disease Using Deep Learn...   
9998        9998  Diosgenin Glucoside Protects against Spinal Co...   
9999        9999  Omics Approaches for Engineering Wheat Product...   

                                               abstract  year  
0     Background Niemann-Pick disease type C (NPC) i...  2

In [24]:
# Extract only the 'abstract' column and drop others
abstracts = dataset['abstract'].dropna().reset_index(drop=True)

# Display a few samples to verify
print(abstracts)

0       Background Niemann-Pick disease type C (NPC) i...
1       Background Metformin toxicity is well known to...
2       Background Measuring service use and costs is ...
3       Background Substance use and delinquency are c...
4       Objectives Intravenous fluids are one of the m...
                              ...                        
9995    The distribution of methylmercury (MeHg) and t...
9996    The purpose of this study was to examine the s...
9997    Infectious disease occurs when a person is inf...
9998    Spinal cord injury (SCI) is a severe traumatic...
9999    Abiotic stresses greatly influenced wheat prod...
Name: abstract, Length: 10000, dtype: object


## **1. TurfTopic Default model and configuration**
  - Use "ModernBERT" to extract embeddings
  - Use Top2Vec to topic modelling ans clustering

In [25]:
# Import topic clustring required libraries
from sentence_transformers import SentenceTransformer
from turftopic import Top2Vec
from turftopic import BERTopic
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [26]:
# Load ModernBERT tokenizer and model from Hugging Face
MODEL_NAME = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

In [27]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [28]:
# # Function to get embeddings for a list of texts
# def get_embeddings(texts, tokenizer, model):
#     model.eval()
#     embeddings = []

#     with torch.no_grad():
#         for text in texts:
#             inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
#             outputs = model(**inputs)
#             # Use [CLS] token embedding (first token)
#             cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
#             embeddings.append(cls_embedding)

#     return embeddings

# Function to get embeddings for a list of texts
def get_embeddings(texts, tokenizer, model):
    model.eval()
    device = next(model.parameters()).device  # ensures model and inputs are on same device
    embeddings = []

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}  # move inputs to GPU
            outputs = model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()  # back to CPU
            embeddings.append(cls_embedding)

    return embeddings

In [29]:
# Wrap the texts with tqdm for progress visualization
abstracts = abstracts
abstracts_with_progress = tqdm(abstracts, desc="Embedding abstracts")

# Call the function with tqdm-wrapped list
abstract_embeddings = get_embeddings(abstracts_with_progress, tokenizer, model)

Embedding abstracts: 100%|██████████| 10000/10000 [07:00<00:00, 23.76it/s]


In [ ]:
# # save abstract_embeddings into csv file
# np.savetxt("abstract_embeddings.csv", abstract_embeddings, delimiter=",")

In [30]:
# Show shape of the first embedding
len(abstract_embeddings), abstract_embeddings[0].shape

(10000, (768,))

In [31]:
# Show embeddings matrix and Check the dimention of each eambeding
embeddings = abstract_embeddings
print(embeddings[:10],"\n\n")

[array([ 3.04976970e-01, -2.08698779e-01, -1.88740358e-01, -2.54587710e-01,
       -4.97646630e-01, -3.29330951e-01, -1.26483965e+00, -8.18425775e-01,
        9.52512026e-01, -9.62196589e-01, -3.15318763e-01,  7.95341611e-01,
       -1.11009145e+00, -3.51078779e-01,  1.54546567e-03, -4.94087994e-01,
       -9.92672324e-01,  5.06244183e-01,  3.00711483e-01,  6.64678156e-01,
        3.30275863e-01, -4.88814265e-01,  4.32895988e-01,  4.71427470e-01,
        4.21916306e-01, -5.02381437e-02, -1.89794481e-01,  7.15767860e-01,
        1.32419765e-01,  9.41680789e-01, -1.50588346e+00,  1.40471971e+00,
       -4.76787947e-02, -5.28304756e-01, -5.52660584e-01,  5.29332817e-01,
        6.08934104e-01,  7.10952044e-01, -4.13420379e-01,  1.17295347e-01,
        3.41138512e-01, -1.92110583e-01,  6.54639304e-02,  2.91758627e-01,
       -2.56493594e-02,  1.01304591e+00,  2.84956731e-02,  3.70313376e-01,
       -5.14247894e-01, -2.82544553e-01,  1.94323397e+00, -4.74122733e-01,
       -5.83610952e-01, 

In [ ]:
# Training model (Uses HDBSCAN and umap)
model = Top2Vec(encoder=encoder, random_state=42)
topic_data = model.prepare_topic_data(abstracts, embeddings=embeddings)

In [ ]:
model.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                                      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       -1 │ metabolomic, metabolomics, biomarkers, proteomics, proteomic, pharmacologically, pharmacological,    │
│          │ biomarker, pharmacology, metabolome                                                                  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │ retinopathy, retinal, intraocular, glaucoma, cataract, ocular, retina, ophthalmic, corneal, macular  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ implant, implants, osteoclasts, osteoblasts, osteogenic, osteoclast, implantation, implanted, bone,  │
│          │ osteoblast                                                                                           │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ dental, tooth, oral, teeth, periodontitis, periodontal, dent, caries, orally, hygiene                │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ periodontitis, periodontal, gingival, dental, cytokine, oral, inflammation, cytokines, tooth,        │
│          │ interleukin                                                                                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ dental, teeth, tooth, periodontal, dent, enamel, periodontitis, maxillary, resin, oral               │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        5 │ judgments, integrity, ethics, inherent, normative, judgment, moral, interpretation, interpretations, │
│          │ necessitates                                                                                         │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        6 │ pollutants, epidemiology, pollution, epidemiological, epidemiologic, pollutant, asthma, polluted,    │
│          │ toxicity, inhaled                                                                                    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        7 │ microscopy, microscope, fluorescence, imaging, immunofluorescence, subcellular, proteomics,          │
│          │ intracellular, intercellular, nanoscale                                                              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        8 │ tuberculosis, mycobacterium, mycobacterial, tb, antibacterial, bactericidal, pathogen, bacteremia,   │
│          │ pneumococcal, pathogens                                                                              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        9 │ classifiers, classifying, classification, classify, supervised, classifier, cnn, bioinformatics,     │
│          │ biomarker, datasets                                                                                  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       10 │ ligands, ligand, compounds, synthesized, molecule, synthesis, catalysis, molecular, aromatic,        │
│          │ catalyzed                                  

In [ ]:
# Cluster model hierarchy
model.hierarchy.cut(3).plot_tree()

In [ ]:
# Merging topics to reduce number of topics
model.reduce_topics(n_reduce_to=25)
print(model.hierarchy.cut(3))

Root: 
├── -1: metabolomic, metabolomics, biomarkers, proteomics, proteomic, pharmacologically, 
│   pharmacological, biomarker, 
│   pharmacology, metabolome
├── 11: sensor, sensing, sensors, accelerometers, gps, accelerometer, iot, tracking, monitoring, 
│   802
├── 104: species, speciation, ecology, phenotypic, phylogenies, biodiversity, fauna, phylogenetic, 
│   taxonomic, 
│   ecological
│   ├── 42: pollination, flowering, plants, pollen, floral, phenotypic, botanical, speciation, flower,
│   │   cultivars
│   └── 43: species, ecology, speciation, fauna, phylogenies, biodiversity, phenotypic, phylogenetic,
│       taxonomic, ecological
├── 113: phytochemical, phytochemicals, flavonoids, antioxidants, antioxidant, antioxidative, 
│   metabolites, phenolics, 
│   flavonoid, herbal
│   ├── 52: phytochemicals, metabolites, antimicrobials, phytochemical, alkaloids, bioactivities, 
│   │   alkaloid, biosynthesis, 
│   │   biogeochemical, biochemical
│   └── 53: phytochemical, phytochemi

In [ ]:
# Model hierarchy after merging topics
fig = model.hierarchy[156].plot_tree()
fig.show()

In [ ]:
# We will reset the hierarchy, so that we can see all topics at once.
model.reset_topics()
fig = model.plot_clusters_datamapplot(hover_text=dataset["title"])
fig.show()

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_data.py:258: UserWarning:

Numerical issues were encountered when centering the data and might not be solved. Dataset may contain too large values. You may need to prescale your features.

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_data.py:277: UserWarning:

Numerical issues were encountered when scaling the data and might not be solved. The standard deviation of the data is probably very close to 0. 

